# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imalik-7/Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Lane decision

I am confirming my lane: **Refresh / Content Opportunity Scoring**.

My baseline asks a simple question:

**Which already-visible pages appear to be getting fewer clicks than similar-position pages and therefore deserve human review first?**

I will check two signals before trusting this rule:

1. **CTR versus position** — this is linked to FlyRank's CTR-fix reasoning. A page's CTR should not be judged without considering where it ranks.
2. **Search volume / impressions** — this is linked to quick-win prioritization. A weakness on a page with meaningful visibility matters more than the same weakness on a page with almost no exposure.

### My one rule

A page becomes a review candidate when:

- it has at least 500 feature-window impressions,
- its average position is between 1 and 20,
- and its CTR is at least 20% below the median CTR of other pages in the same position tier.

The score increases with both visibility and the size of the CTR gap.

**Reason code:** `visible_low_ctr_for_position`

**Action label:** `review_title_meta_and_intent`

The rule uses only information from March 1–15. Future-window impressions and the decline proxy are never inputs to the score.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

In [2]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import whoami

# -----------------------------
# Hugging Face authentication
# -----------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add it in Colab's Secrets panel first."
    )

account = whoami(token=HF_TOKEN)

print("Hugging Face connected.")
print("Account:", account["name"])

# -----------------------------
# DuckDB connection
# -----------------------------

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{safe_token}'
)
""")

MARCH_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

MARCH = f"read_parquet('{MARCH_PATH}')"

print("DuckDB ready.")
print("Development month: March 2026")

Hugging Face connected.
Account: imalik7
DuckDB ready.
Development month: March 2026


In [3]:
schema = con.sql(f"""
DESCRIBE SELECT *
FROM {MARCH}
""").df()

required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

missing = [
    col for col in required_columns
    if col not in schema["column_name"].tolist()
]

print("Missing required columns:", missing)

assert not missing, (
    "Some required columns are missing. "
    "Check the schema before continuing."
)

print("PASS: all required columns exist.")

Missing required columns: []
PASS: all required columns exist.


In [4]:
analysis_df = con.sql(f"""
WITH page_windows AS (

    SELECT
        client_hash_id,
        content_hash_id,

        -- --------------------
        -- FEATURE WINDOW
        -- March 1-15
        -- --------------------

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS impressions_feature15,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS clicks_feature15,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                     AND gsc_avg_position > 0
                THEN gsc_avg_position * COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                         AND gsc_avg_position > 0
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ),
            0
        ) AS avg_position_feature15,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                     AND COALESCE(gsc_impressions, 0) > 0
                THEN report_date
            END
        ) AS active_impression_days_feature15,

        -- --------------------
        -- OUTCOME WINDOW
        -- March 16-30
        -- Used ONLY for evaluation
        -- --------------------

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-16'
                                     AND DATE '2026-03-30'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS impressions_outcome15

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    *,

    100.0 * clicks_feature15
    / NULLIF(impressions_feature15, 0)
    AS ctr_feature15,

    CASE
        WHEN impressions_outcome15
             < 0.80 * impressions_feature15
        THEN 1
        ELSE 0
    END AS is_declining_proxy

FROM page_windows

WHERE impressions_feature15 >= 100

""").df()

print("Analysis rows:", len(analysis_df))
print(
    "Declining proxy rate:",
    round(analysis_df["is_declining_proxy"].mean(), 3)
)

display(analysis_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Analysis rows: 77540
Declining proxy rate: 0.327


,client_hash_id,content_hash_id,impressions_feature15,clicks_feature15,avg_position_feature15,active_impression_days_feature15,impressions_outcome15,ctr_feature15,is_declining_proxy
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,4.386946,15,676.0,0.466200,0
1,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,5.265924,15,741.0,0.159236,0
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,4.144531,15,1402.0,0.703125,0
3,client_73cda7b4e4f265ea,content_712c365258cee05c,2531.0,10.0,4.861320,15,3258.0,0.395101,0
4,client_73cda7b4e4f265ea,content_3dba50ae010f3f30,243.0,1.0,14.534979,15,113.0,0.411523,1


In [5]:
# ============================================================
# SIGNAL CHECK 1
# CTR versus search position
# ============================================================

ctr_audit = analysis_df[
    analysis_df["avg_position_feature15"].between(1, 20)
].copy()

# Position tiers
ctr_audit["position_tier"] = pd.cut(
    ctr_audit["avg_position_feature15"],
    bins=[0, 3, 10, 20],
    labels=[
        "position_1_3",
        "position_4_10",
        "position_11_20",
    ],
    include_lowest=True,
)

# Expected CTR = median CTR among pages in same position tier
expected_ctr = (
    ctr_audit
    .groupby(
        "position_tier",
        observed=True
    )["ctr_feature15"]
    .transform("median")
)

ctr_audit["expected_ctr"] = expected_ctr

ctr_audit["ctr_vs_expected"] = (
    ctr_audit["ctr_feature15"]
    /
    ctr_audit["expected_ctr"].replace(0, np.nan)
)

# Bucket the relative CTR
ctr_audit["ctr_bucket"] = pd.cut(
    ctr_audit["ctr_vs_expected"],
    bins=[
        -np.inf,
        0.50,
        0.80,
        1.20,
        np.inf,
    ],
    labels=[
        "very_low",
        "low",
        "near_expected",
        "above_expected",
    ],
)

ctr_bucket_table = (
    ctr_audit
    .groupby(
        "ctr_bucket",
        observed=True
    )
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr_feature15", "median"),
        median_expected_ctr=("expected_ctr", "median"),
        decline_rate=("is_declining_proxy", "mean"),
    )
    .reset_index()
)

ctr_bucket_table["decline_rate"] = (
    100 * ctr_bucket_table["decline_rate"]
).round(1)

print("SIGNAL 1 — CTR VS POSITION")
display(ctr_bucket_table)

# ------------------------------------------------------------
# Data-driven verdict
# ------------------------------------------------------------

table_indexed = ctr_bucket_table.set_index("ctr_bucket")

if (
    "very_low" not in table_indexed.index
    or "above_expected" not in table_indexed.index
):
    CTR_VERDICT = "FALSE"

else:
    very_low_rate = table_indexed.loc[
        "very_low", "decline_rate"
    ]

    above_rate = table_indexed.loc[
        "above_expected", "decline_rate"
    ]

    difference = very_low_rate - above_rate

    if difference >= 5:
        CTR_VERDICT = "CONFIRMED"

    elif difference <= -5:
        CTR_VERDICT = "OPPOSITE"

    else:
        CTR_VERDICT = "MIXED"

print("VERDICT:", CTR_VERDICT)

SIGNAL 1 — CTR VS POSITION


,ctr_bucket,n,median_ctr,median_expected_ctr,decline_rate
0,very_low,24770,0.000000,0.189036,38.9
1,low,3526,0.132882,0.189036,37.3
2,near_expected,4714,0.197093,0.189036,31.7
3,above_expected,28519,0.491400,0.189036,23.6


VERDICT: CONFIRMED


In [6]:
# ============================================================
# SIGNAL CHECK 2
# Search visibility / volume
# ============================================================

volume_audit = analysis_df.copy()

volume_audit["volume_bucket"] = pd.cut(
    volume_audit["impressions_feature15"],
    bins=[
        99,
        249,
        499,
        999,
        np.inf,
    ],
    labels=[
        "100_249",
        "250_499",
        "500_999",
        "1000_plus",
    ],
)

# Audit-only future outcome:
# this is NEVER used as a baseline feature.
volume_audit["observed_impression_loss"] = (
    volume_audit["impressions_feature15"]
    -
    volume_audit["impressions_outcome15"]
).clip(lower=0)

volume_bucket_table = (
    volume_audit
    .groupby(
        "volume_bucket",
        observed=True
    )
    .agg(
        n=("content_hash_id", "size"),
        avg_feature_impressions=(
            "impressions_feature15",
            "mean"
        ),
        avg_observed_loss=(
            "observed_impression_loss",
            "mean"
        ),
        decline_rate=(
            "is_declining_proxy",
            "mean"
        ),
    )
    .reset_index()
)

volume_bucket_table[
    "avg_feature_impressions"
] = volume_bucket_table[
    "avg_feature_impressions"
].round(1)

volume_bucket_table[
    "avg_observed_loss"
] = volume_bucket_table[
    "avg_observed_loss"
].round(1)

volume_bucket_table[
    "decline_rate"
] = (
    100 *
    volume_bucket_table[
        "decline_rate"
    ]
).round(1)

print("SIGNAL 2 — SEARCH VOLUME")
display(volume_bucket_table)

# ------------------------------------------------------------
# Data-driven verdict
# ------------------------------------------------------------

volume_indexed = volume_bucket_table.set_index(
    "volume_bucket"
)

if (
    "100_249" not in volume_indexed.index
    or "1000_plus" not in volume_indexed.index
):
    VOLUME_VERDICT = "FALSE"

else:
    low_loss = volume_indexed.loc[
        "100_249",
        "avg_observed_loss"
    ]

    high_loss = volume_indexed.loc[
        "1000_plus",
        "avg_observed_loss"
    ]

    if high_loss > low_loss * 1.5:
        VOLUME_VERDICT = "CONFIRMED"

    elif high_loss < low_loss:
        VOLUME_VERDICT = "OPPOSITE"

    else:
        VOLUME_VERDICT = "MIXED"

print("VERDICT:", VOLUME_VERDICT)

SIGNAL 2 — SEARCH VOLUME


,volume_bucket,n,avg_feature_impressions,avg_observed_loss,decline_rate
0,100_249,20468,163.1,28.3,32.3
1,250_499,15256,358.7,62.1,33.1
2,500_999,14828,722.5,112.7,31.1
3,1000_plus,26988,3930.8,688.3,33.6


VERDICT: CONFIRMED


### Signal verdicts

The notebook prints the one-word verdict for each signal from the observed bucket tables rather than forcing a positive result.

- **CTR versus position:** see the `VERDICT` printed above.
- **Search volume:** see the `VERDICT` printed above.

A CONFIRMED result supports using the signal in the baseline. A MIXED or OPPOSITE result is also useful because it shows where the hand-written rule is weak and gives the Week-5 model something honest to improve on.

The outcome-window fields used in these audit tables are for retrospective signal checking only. They are not used as inputs to the baseline score.

## 2. Build the ranked queue (writes the CSV)

## Baseline rule

I will now freeze one transparent rule.

A page is prioritized when it:

1. has meaningful visibility,
2. ranks within positions 1–20,
3. and has CTR at least 20% below the typical CTR of pages in the same position tier.

The score is intentionally simple:

**baseline score = impressions × CTR-gap severity**

This keeps the baseline understandable. A Week-5 model only earns its place if it can beat this frozen rule under the same evaluation setup.

**Reason code:** `visible_low_ctr_for_position`

**Action:** `review_title_meta_and_intent`

In [7]:
# ============================================================
# BUILD THE BASELINE RANKED QUEUE
# ============================================================

queue = analysis_df.copy()

# ------------------------------------------------------------
# Position tiers
# ------------------------------------------------------------

queue["position_tier"] = pd.cut(
    queue["avg_position_feature15"],
    bins=[0, 3, 10, 20, np.inf],
    labels=[
        "position_1_3",
        "position_4_10",
        "position_11_20",
        "position_21_plus",
    ],
    include_lowest=True,
)

# ------------------------------------------------------------
# Expected CTR from FEATURE-WINDOW data only
# ------------------------------------------------------------

expected_ctr_by_tier = (
    queue[
        queue["avg_position_feature15"]
        .between(1, 20)
    ]
    .groupby(
        "position_tier",
        observed=True
    )["ctr_feature15"]
    .median()
    .to_dict()
)

queue["expected_ctr_feature15"] = (
    queue["position_tier"]
    .map(expected_ctr_by_tier)
    .astype(float)
)

# ------------------------------------------------------------
# CTR-gap severity
# 0 = at/above expected
# larger value = further below expected
# ------------------------------------------------------------

queue["ctr_gap_severity"] = (
    1
    -
    (
        queue["ctr_feature15"]
        /
        queue[
            "expected_ctr_feature15"
        ].replace(0, np.nan)
    )
).clip(lower=0)

# ------------------------------------------------------------
# ONE hand-written rule
# ------------------------------------------------------------

queue["rule_flag"] = (
    (queue["impressions_feature15"] >= 500)
    &
    (
        queue["avg_position_feature15"]
        .between(1, 20)
    )
    &
    (
        queue["ctr_gap_severity"] >= 0.20
    )
)

# ------------------------------------------------------------
# Transparent baseline score
# ------------------------------------------------------------

queue["baseline_action_score"] = np.where(
    queue["rule_flag"],
    queue["impressions_feature15"]
    *
    queue["ctr_gap_severity"],
    0.0,
)

# ------------------------------------------------------------
# ONE reason code
# ------------------------------------------------------------

queue["reason_code"] = np.where(
    queue["rule_flag"],
    "visible_low_ctr_for_position",
    "",
)

# ------------------------------------------------------------
# Action label
# ------------------------------------------------------------

queue["action_label"] = np.where(
    queue["rule_flag"],
    "review_title_meta_and_intent",
    "monitor",
)

# ------------------------------------------------------------
# Rank
# ------------------------------------------------------------

queue = (
    queue
    .sort_values(
        "baseline_action_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

queue["baseline_rank"] = (
    np.arange(len(queue)) + 1
)

flagged_count = int(
    queue["rule_flag"].sum()
)

print("Rows in queue:", len(queue))
print("Rows flagged by rule:", flagged_count)

display(
    queue[
        [
            "baseline_rank",
            "content_hash_id",
            "impressions_feature15",
            "avg_position_feature15",
            "ctr_feature15",
            "expected_ctr_feature15",
            "ctr_gap_severity",
            "baseline_action_score",
            "reason_code",
            "action_label",
        ]
    ].head(10)
)

Rows in queue: 77540
Rows flagged by rule: 11552


,baseline_rank,content_hash_id,impressions_feature15,avg_position_feature15,ctr_feature15,expected_ctr_feature15,ctr_gap_severity,baseline_action_score,reason_code,action_label
0,1,content_34a70fea29d15f24,73639.0,2.948003,0.024444,0.260558,0.906188,66730.752036,visible_low_ctr_for_position,review_title_meta_and_intent
1,2,content_7c6373141eae744a,86860.0,5.953765,0.058715,0.189036,0.689397,59881.000000,visible_low_ctr_for_position,review_title_meta_and_intent
2,3,content_8e1334d6356668e3,58553.0,4.753471,0.001708,0.189036,0.990965,58024.000000,visible_low_ctr_for_position,review_title_meta_and_intent
3,4,content_945d6ff91386c817,49314.0,8.416089,0.004056,0.189036,0.978546,48256.000000,visible_low_ctr_for_position,review_title_meta_and_intent
4,5,content_65c75874a23fca87,55680.0,6.682004,0.026940,0.189036,0.857489,47745.000000,visible_low_ctr_for_position,review_title_meta_and_intent
5,6,content_f6116743b00afc2d,49619.0,9.817671,0.016123,0.189036,0.914710,45387.000000,visible_low_ctr_for_position,review_title_meta_and_intent
6,7,content_1642f339bd6e7c8d,52378.0,4.651991,0.034366,0.189036,0.818206,42856.000000,visible_low_ctr_for_position,review_title_meta_and_intent
7,8,content_36fc1ee501ec072d,46199.0,5.300223,0.023810,0.189036,0.874045,40380.000000,visible_low_ctr_for_position,review_title_meta_and_intent
8,9,content_cd3d932d4e1c8db0,34191.0,8.290661,0.005849,0.189036,0.969056,33133.000000,visible_low_ctr_for_position,review_title_meta_and_intent
9,10,content_62673eea26c31c17,49386.0,5.729114,0.070870,0.189036,0.625096,30871.000000,visible_low_ctr_for_position,review_title_meta_and_intent


In [8]:
# ============================================================
# BASELINE EVALUATION
# Label is used ONLY here for retrospective evaluation.
# It was not used to calculate the score.
# ============================================================

def precision_at_k(df, k):
    top = df.head(min(k, len(df)))

    if len(top) == 0:
        return np.nan

    return top[
        "is_declining_proxy"
    ].mean()


base_rate = queue[
    "is_declining_proxy"
].mean()

precision_10 = precision_at_k(
    queue,
    10
)

precision_50 = precision_at_k(
    queue,
    50
)

print(
    f"Base decline rate: "
    f"{base_rate:.3f}"
)

print(
    f"Baseline Precision@10: "
    f"{precision_10:.3f}"
)

print(
    f"Baseline Precision@50: "
    f"{precision_50:.3f}"
)

Base decline rate: 0.327
Baseline Precision@10: 0.600
Baseline Precision@50: 0.500


In [9]:
# ============================================================
# WRITE REQUIRED OUTPUT CSV
# ============================================================

OUTPUT_DIR = Path(
    "work/outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CSV_PATH = (
    OUTPUT_DIR
    /
    "baseline_action_score.csv"
)

# Do NOT export the future outcome or target into
# the deployable action queue.
queue_export = queue[
    [
        "baseline_rank",
        "client_hash_id",
        "content_hash_id",
        "impressions_feature15",
        "clicks_feature15",
        "ctr_feature15",
        "avg_position_feature15",
        "active_impression_days_feature15",
        "position_tier",
        "expected_ctr_feature15",
        "ctr_gap_severity",
        "baseline_action_score",
        "reason_code",
        "action_label",
    ]
].copy()

queue_export.to_csv(
    CSV_PATH,
    index=False
)

print(
    "CSV written successfully:"
)

print(CSV_PATH)

print(
    "Rows written:",
    len(queue_export)
)

CSV written successfully:
work/outputs/baseline_action_score.csv
Rows written: 77540


In [10]:
metrics = {
    "lane": (
        "Refresh / Content Opportunity Scoring"
    ),
    "development_month": "2026-03",
    "feature_window": (
        "2026-03-01 to 2026-03-15"
    ),
    "outcome_window": (
        "2026-03-16 to 2026-03-30"
    ),
    "rule": (
        "impressions>=500, position 1-20, "
        "CTR >=20% below same-tier median"
    ),
    "reason_code": (
        "visible_low_ctr_for_position"
    ),
    "action_label": (
        "review_title_meta_and_intent"
    ),
    "rows": int(len(queue)),
    "flagged_rows": flagged_count,
    "base_rate": float(base_rate),
    "precision_at_10": float(
        precision_10
    ),
    "precision_at_50": float(
        precision_50
    ),
    "ctr_signal_verdict": CTR_VERDICT,
    "volume_signal_verdict": (
        VOLUME_VERDICT
    ),
}

JSON_PATH = (
    OUTPUT_DIR
    /
    "w04_baseline_metrics.json"
)

with open(
    JSON_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metrics,
        f,
        indent=2,
    )

print(
    "Metrics JSON written:",
    JSON_PATH,
)

print(
    json.dumps(
        metrics,
        indent=2,
    )
)

Metrics JSON written: work/outputs/w04_baseline_metrics.json
{
  "lane": "Refresh / Content Opportunity Scoring",
  "development_month": "2026-03",
  "feature_window": "2026-03-01 to 2026-03-15",
  "outcome_window": "2026-03-16 to 2026-03-30",
  "rule": "impressions>=500, position 1-20, CTR >=20% below same-tier median",
  "reason_code": "visible_low_ctr_for_position",
  "action_label": "review_title_meta_and_intent",
  "rows": 77540,
  "flagged_rows": 11552,
  "base_rate": 0.32667010575187,
  "precision_at_10": 0.6,
  "precision_at_50": 0.5,
  "ctr_signal_verdict": "CONFIRMED",
  "volume_signal_verdict": "CONFIRMED"
}


### Required Top-10 review

The current assignment requires ten reviewed rows, so I review the top ten baseline candidates below.

For each row I record:

- the suggested action,
- why the rule ranked it highly,
- and what evidence could make the recommendation wrong.

These are review candidates, not automatic edit instructions.

In [11]:
# ============================================================
# TOP-10 HUMAN REVIEW
# ============================================================

flagged_queue = queue[
    queue["baseline_action_score"] > 0
].copy()

top10 = flagged_queue.head(10).copy()

if len(top10) < 10:
    print(
        "WARNING:",
        "Fewer than 10 rows were flagged."
    )


def wrong_reason(row):

    if (
        row[
            "active_impression_days_feature15"
        ] < 8
    ):
        return (
            "The signal may be unstable because "
            "the page had too few active days."
        )

    elif (
        row[
            "avg_position_feature15"
        ] <= 3
    ):
        return (
            "SERP features, brand intent, or query "
            "mix may explain low CTR even with a "
            "strong position."
        )

    elif (
        row[
            "impressions_feature15"
        ] < 750
    ):
        return (
            "The apparent CTR gap may be noise at "
            "this moderate volume."
        )

    else:
        return (
            "Seasonality, query mix, or a SERP "
            "change could explain the CTR gap "
            "without the page itself needing an edit."
        )


review_rows = []

for _, row in top10.iterrows():

    why = (
        f"{int(row['impressions_feature15']):,} "
        f"impressions; position "
        f"{row['avg_position_feature15']:.1f}; "
        f"CTR {row['ctr_feature15']:.2f}% vs "
        f"tier expectation "
        f"{row['expected_ctr_feature15']:.2f}%."
    )

    review_rows.append(
        {
            "rank": int(
                row["baseline_rank"]
            ),
            "content_hash_id": (
                row["content_hash_id"]
            ),
            "action": (
                row["action_label"]
            ),
            "why_it_is_here": why,
            "what_would_make_it_wrong": (
                wrong_reason(row)
            ),
            "proxy_outcome": (
                "declined"
                if row[
                    "is_declining_proxy"
                ] == 1
                else "did_not_decline"
            ),
        }
    )


top10_review = pd.DataFrame(
    review_rows
)

display(top10_review)

,rank,content_hash_id,action,why_it_is_here,what_would_make_it_wrong,proxy_outcome
0,1,content_34a70fea29d15f24,review_title_meta_and_intent,"73,639 impressions; position 2.9; CTR 0.02% vs...","SERP features, brand intent, or query mix may ...",did_not_decline
1,2,content_7c6373141eae744a,review_title_meta_and_intent,"86,860 impressions; position 6.0; CTR 0.06% vs...","Seasonality, query mix, or a SERP change could...",declined
2,3,content_8e1334d6356668e3,review_title_meta_and_intent,"58,553 impressions; position 4.8; CTR 0.00% vs...","Seasonality, query mix, or a SERP change could...",did_not_decline
3,4,content_945d6ff91386c817,review_title_meta_and_intent,"49,314 impressions; position 8.4; CTR 0.00% vs...","Seasonality, query mix, or a SERP change could...",declined
4,5,content_65c75874a23fca87,review_title_meta_and_intent,"55,680 impressions; position 6.7; CTR 0.03% vs...","Seasonality, query mix, or a SERP change could...",declined
5,6,content_f6116743b00afc2d,review_title_meta_and_intent,"49,619 impressions; position 9.8; CTR 0.02% vs...","Seasonality, query mix, or a SERP change could...",did_not_decline
6,7,content_1642f339bd6e7c8d,review_title_meta_and_intent,"52,378 impressions; position 4.7; CTR 0.03% vs...","Seasonality, query mix, or a SERP change could...",declined
7,8,content_36fc1ee501ec072d,review_title_meta_and_intent,"46,199 impressions; position 5.3; CTR 0.02% vs...","Seasonality, query mix, or a SERP change could...",declined
8,9,content_cd3d932d4e1c8db0,review_title_meta_and_intent,"34,191 impressions; position 8.3; CTR 0.01% vs...","Seasonality, query mix, or a SERP change could...",did_not_decline
9,10,content_62673eea26c31c17,review_title_meta_and_intent,"49,386 impressions; position 5.7; CTR 0.07% vs...","Seasonality, query mix, or a SERP change could...",declined


In [12]:
for _, row in top10_review.iterrows():

    print(
        f"#{row['rank']} | "
        f"Action: {row['action']} | "
        f"Why: {row['why_it_is_here']} | "
        f"Wrong if: "
        f"{row['what_would_make_it_wrong']}"
    )

#1 | Action: review_title_meta_and_intent | Why: 73,639 impressions; position 2.9; CTR 0.02% vs tier expectation 0.26%. | Wrong if: SERP features, brand intent, or query mix may explain low CTR even with a strong position.
#2 | Action: review_title_meta_and_intent | Why: 86,860 impressions; position 6.0; CTR 0.06% vs tier expectation 0.19%. | Wrong if: Seasonality, query mix, or a SERP change could explain the CTR gap without the page itself needing an edit.
#3 | Action: review_title_meta_and_intent | Why: 58,553 impressions; position 4.8; CTR 0.00% vs tier expectation 0.19%. | Wrong if: Seasonality, query mix, or a SERP change could explain the CTR gap without the page itself needing an edit.
#4 | Action: review_title_meta_and_intent | Why: 49,314 impressions; position 8.4; CTR 0.00% vs tier expectation 0.19%. | Wrong if: Seasonality, query mix, or a SERP change could explain the CTR gap without the page itself needing an edit.
#5 | Action: review_title_meta_and_intent | Why: 55,680 i

## Weak picks and leakage check

A weak pick is a page that the baseline ranked highly but that did not match my later decline proxy.

That does not automatically prove the CTR-review action was useless. My baseline is identifying a visible CTR opportunity, while my current proxy measures later impression decline. Those ideas overlap but are not identical.

Still, these disagreements are valuable because they show where the rule is imperfect and what the Week-5 model must improve.

The baseline itself must remain completely free of outcome-window measurements and label-derived inputs.

In [13]:
# ============================================================
# WEAK PICKS
# ============================================================

top10_with_outcome = top10[
    [
        "baseline_rank",
        "content_hash_id",
        "baseline_action_score",
        "impressions_feature15",
        "avg_position_feature15",
        "ctr_feature15",
        "expected_ctr_feature15",
        "is_declining_proxy",
    ]
].copy()

weak_picks = top10_with_outcome[
    top10_with_outcome[
        "is_declining_proxy"
    ] == 0
].copy()

print(
    "Weak picks in top 10:",
    len(weak_picks)
)

display(weak_picks)

Weak picks in top 10: 4


,baseline_rank,content_hash_id,baseline_action_score,impressions_feature15,avg_position_feature15,ctr_feature15,expected_ctr_feature15,is_declining_proxy
0,1,content_34a70fea29d15f24,66730.752036,73639.0,2.948003,0.024444,0.260558,0
2,3,content_8e1334d6356668e3,58024.000000,58553.0,4.753471,0.001708,0.189036,0
5,6,content_f6116743b00afc2d,45387.000000,49619.0,9.817671,0.016123,0.189036,0
8,9,content_cd3d932d4e1c8db0,33133.000000,34191.0,8.290661,0.005849,0.189036,0


In [14]:
# ============================================================
# LEAKAGE CHECK
# ============================================================

RULE_INPUTS = {
    "impressions_feature15",
    "ctr_feature15",
    "avg_position_feature15",
    "expected_ctr_feature15",
    "ctr_gap_severity",
}

FORBIDDEN_INPUTS = {
    "impressions_outcome15",
    "is_declining_proxy",
    "observed_impression_loss",
}

leaks = (
    RULE_INPUTS
    &
    FORBIDDEN_INPUTS
)

print(
    "Forbidden fields used by rule:",
    leaks
)

assert not leaks, (
    "LEAKAGE DETECTED."
)

print(
    "PASS: baseline score uses "
    "pre-decision information only."
)

print(
    "Future outcome and label were used "
    "only for retrospective evaluation."
)

Forbidden fields used by rule: set()
PASS: baseline score uses pre-decision information only.
Future outcome and label were used only for retrospective evaluation.


### Week-4 conclusion

I am keeping the **Refresh / Content Opportunity Scoring** lane.

My baseline is intentionally simple and frozen: prioritize visible pages whose CTR is meaningfully below pages at similar search positions.

The rule produces one score, one reason code, and one review action. Its weaknesses are visible in the top-10 review rather than hidden.

The Week-5 model now has a clear job: beat this baseline on the same slice and evaluation setup without using any future-window or label-derived information.

## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.